# Setup

In [1]:
#essential reticulate functions (must run first)
Sys.setenv(RETICULATE_PYTHON="/home/luca/.conda/envs/reticulate.magic/bin/python")
library(reticulate)
reticulate::use_python("/home/luca/.conda/envs/reticulate.magic/bin/python")
reticulate::use_condaenv("/home/luca/.conda/envs/reticulate.magic")

In [2]:
pacman::p_load(dplyr, stringr, data.table, tidyr, plyr,
               pheatmap, colorRamps, gridExtra, ggplot2, ggrepel, RColorBrewer, ComplexHeatmap, ggbreak,
               Rmagic, phateR,
               Seurat,logr)

In [3]:
# Setup folders
Seurat.obj = "/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/nov2022/Nov22_objects_final/20221216_finalObjects_finalFiles/RNA/SeuratObjects/20230518_RNA_FiltMin20exceptBetaLate_CellChat_nPODids.rds"
Log.dir = "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/"
Assets.dir = "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/Assets/"
Seurat.SCT = "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/Assets/20230518_RNA_FiltMin20exceptBetaLate_CellChat_nPODids.SCT.rds"

# Options and log
options(stringsAsFactors = FALSE)
options("logr.on" = TRUE, "logr.notes" = TRUE)
options("logr.autolog" = TRUE)
options("logr.compact" = TRUE)
options("logr.traceback" = TRUE)

In [6]:
# Open Log
log_open(Log.dir)

[1] "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/log/.log"

# SCT transformation

In [7]:
seurat_object <- readRDS(Seurat.obj)

In [9]:
levels(factor(seurat_object$nPOD_ID))

[1] "6197" "6220" "6228" "6229" "6234" "6236" "6247" "6264" "6267" "6282"
[11] "6310" "6339" "6362" "6366" "6375" "6380" "6393" "6401" "6405" "6418"
[21] "6424" "6429" "6431" "6450" "6456" "6459" "6479" "6505" "6512" "6520"
[31] "6521"

In [10]:
## Normalize metadata columns:
# seurat_object$samples = seurat_object$library
seurat_object$condition = seurat_object$Condition_subtype
seurat_object$celltype = seurat_object$CurrentObjLabels
seurat_object$library = seurat_object$samples
seurat_object$samples = seurat_object$nPOD_ID

In [11]:
table(seurat_object$technology)
seurat_object
seurat_object <- subset(x = seurat_object, subset = technology == "snRNA")
seurat_object
gc(reset = TRUE)


 snRNA 
232198 

An object of class Seurat 
73202 features across 232198 samples within 2 assays 
Active assay: RNA (36601 features, 2000 variable features)
 1 other assay present: RAW_mtx
 3 dimensional reductions calculated: pca, harmony, umap

An object of class Seurat 
73202 features across 232198 samples within 2 assays 
Active assay: RNA (36601 features, 2000 variable features)
 1 other assay present: RAW_mtx
 3 dimensional reductions calculated: pca, harmony, umap

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,5373091,287.0,9531355,509.1,5373091,287.0
Vcells,2740658691,20909.6,3901070338,29762.9,2740658691,20909.6


In [12]:
seurat_object <- SetIdent(seurat_object, value = seurat_object$celltype)

In [13]:
levels(factor(seurat_object$condition))

[1] "Aab"       "ND"        "T1D_early" "T1D_late"

In [14]:
# Check sample numbers are accurate
condition.ls = c("ND", "Aab", "T1D_early", "T1D_late")

group.sizes = data.frame(table(seurat_object$samples, seurat_object$condition))
colnames(group.sizes) = c("sample", "Condition", "Freq")
ND.size = nrow(group.sizes[group.sizes$Condition == condition.ls[1] & group.sizes$Freq > 0,])
AAB.size = nrow(group.sizes[group.sizes$Condition == condition.ls[2] & group.sizes$Freq > 0,])
ETD.size = nrow(group.sizes[group.sizes$Condition == condition.ls[3] & group.sizes$Freq > 0,])
LTD.size = nrow(group.sizes[group.sizes$Condition == condition.ls[4] & group.sizes$Freq > 0,])
ND.size
AAB.size
ETD.size
LTD.size

[1] 10

[1] 9

[1] 7

[1] 5

In [15]:
gc(reset = TRUE)
seurat_object <- SCTransform(seurat_object, assay = "RNA", 
                                   new.assay.name = "RNA.SCT", verbose = FALSE)

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,5379734,287.4,9531355,509.1,5379734,287.4
Vcells,2740441519,20908.0,4681364405,35716.0,2740441519,20908.0


Warning message:
“Keys should be one or more alphanumeric characters followed by an underscore, setting key from rna.sct_ to rnasct_”


In [16]:
saveRDS(seurat_object, file = Seurat.SCT)

In [ ]:
# No need to do the step below as I already have the min20 cells RDS

# Min20 cells RDS

In [ ]:
setwd(Assets.dir)

sample_bcs <- list()
unique_cell_types <- unique(seurat_object$celltype)


###### LIST SAMPLES ########
samples <- levels(factor(seurat_object$samples))


for (sample in samples){
    print(sample)
    good_celltypes <- list()
    obj <- subset(seurat_object, subset = samples == sample)
    obj
    for (celltype in unique_cell_types){
        if (sum(Idents(obj)==celltype) >= 20){
            good_celltypes[[celltype]] <- celltype 
        }
        good <- names(good_celltypes)
    sample_bcs[[sample]] <- row.names(obj[[]][obj[[]]$celltype %in% good,])
    }
}
lapply(sample_bcs, write, "goodBCs.txt", append=TRUE)


#write.table(sample_bcs,'LUCA_goodBCs.txt',col.names=FALSE, quote=FALSE)
toKeep_final <-  file.path("goodBCs.txt")
keep.cells1 <- trimws(scan(toKeep_final, what='', sep='\n'), which='right', whitespace=' ')
keep <- c(keep.cells1)

In [ ]:
print(length(keep))
seurat_object
seurat_object$keep_cells <- (Cells(seurat_object) %in% keep)
seurat_object <- subset(seurat_object, subset=keep_cells==TRUE)
seurat_object


saveRDS(seurat_object, file = Seurat.SCT.min20)